In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :powerlaw

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [powerlaw_model] Fitting chain 2 (tau=59)
[ Info: [powerlaw] iter 1000/1000000 elapsed=5.7s, rate=0.083, mean=[0.800, 0.00096, 0.334, 0.563], std=[0.1194, 0.000332, 0.0174, 0.0797] [ADAPT]
[ Info: [powerlaw] iter 2000/1000000 elapsed=10.6s, rate=0.072, mean=[0.717, 0.00093, 0.341, 0.626], std=[0.1129, 0.000246, 0.0143, 0.0803] [ADAPT]
[ Info: [powerlaw] iter 3000/1000000 elapsed=14.7s, rate=0.064, mean=[0.700, 0.00090, 0.346, 0.659], std=[0.0947, 0.000214, 0.0138, 0.0794] [ADAPT]
[ Info: [powerlaw] iter 4000/1000000 elapsed=18.8s, rate=0.061, mean=[0.697, 0.00089, 0.351, 0.668], std=[0.0823, 0.000192, 0.0144, 0.0702] [ADAPT]
[ Info: [powerlaw] iter 5000/1000000 elapsed=22.8s, rate=0.061, mean=[0.688, 0.00091, 0.356, 0.663], std=[0.0759, 0.000187, 0.0154, 0.0643] [ADAPT]
[ Info: [powerlaw] iter 6000/1000000 elapsed=26.9s, rate=0.060, mean=[0.678, 0.00093, 0.356, 0.660], std=[0.0719, 0.000178, 0.0148, 0.0593] [ADAPT]
[ Info: [powerlaw] iter 7000/1000000 elapsed=31.0s, rate=0.058,